# PIGEON-AI: area-weighted population allocation

Understand the population estimate within complete hexagons. Inputs: authoritative features and original Census/SA1 sources used by the pipeline. Outputs: summaries and a population map. The implementation is in `src/pigeon_ai/population.py`; this notebook does not regenerate or overwrite outputs.

In [ ]:
from pathlib import Path
import sys

# Works when the kernel starts in the project root or notebooks/.
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from pigeon_ai.paths import INTERIM, PROCESSED, CRS

assert (PROCESSED / "sydney_hex_features.gpkg").exists(), "Run python scripts/run_pipeline.py first."


## Original-area denominators

For every SA1–hexagon intersection, contribution = SA1 population × intersection area / **complete original SA1 area**. Selecting relevant SA1s does not clip their geometries. EPSG:7856 supplies metric areas. Uniform population within each SA1 is an approximation; zero, water and sparsely occupied space remain possible. The pipeline rejects missing contextual Census values before allocation.

In [ ]:
features = gpd.read_file(PROCESSED / "sydney_hex_features.gpkg")
assert (features.population_est >= 0).all()
assert features.population_est.notna().all()
recomputed_density = features.population_est / (features.area / 1_000_000)
import numpy as np
assert np.allclose(recomputed_density, features.population_density)
print("Estimated population in complete grid:", round(features.population_est.sum()))
display(features[["population_est", "population_density"]].describe())

The population represented by complete selected hexagons need not equal the sum of populations in the selected SA1s: the two spatial footprints differ. These estimates describe residents in 2021, not visitors or daytime demand.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
features.plot(ax=ax, column="population_est", cmap="YlOrRd", legend=True, linewidth=0)
ax.set(title="Area-weighted estimated residents per hexagon")
ax.set_axis_off()
plt.show()